# Storm tracking over Indonesia

Detects and tracks storms in GPM IMERG half-hourly precipitation, and writes a
catalogue with date and time, total volume, duration, max intensity, centroid,
weighted centroid, start and end location, and the trackline.

**Set the date range in the next cell. Everything else follows from it.**
No local archive is required: by default the data is fetched from NASA GES DISC
with server-side subsetting and cached, so only the chosen period is downloaded.

### Scope, stated up front

This produces a **catalogue**, not a validated product. There is no gauge or radar
comparison for Indonesia in this project, so nothing here should be read as a
statement of accuracy. See `docs/plan.md`.


## 1. Choose the period

One day is about 16 s and 3.9 MB over the network. One month is about 8 min and
121 MB, and gives a 511 MB cube. A month is the online ceiling; beyond it set
`SOURCE = "local"`.


In [ ]:
START   = "2020-12-01"
END     = "2020-12-05"     # inclusive

SOURCE  = None             # None = config default (opendap). Or "local".
PRODUCT = "final"          # final | late | early

# Segmentation. PROMINENCE is the one parameter that decides what counts as a
# distinct storm, in mm/hr. It is NOT scale-free: the storm count roughly halves
# for each doubling. Quote it alongside every number you take from the catalogue.
WET_THRESHOLD = 1.0
PROMINENCE    = 4.0
MIN_VOXELS    = 6

## 2. Setup

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent / "src"))

import numpy as np, pandas as pd, matplotlib.pyplot as plt

from imerg_io import load_range
from segment import label_ccl, label_watershed, percolation_share
from storms import storm_table, track_wkt
import catalogue

pd.set_option("display.width", 180)
plt.rcParams["figure.dpi"] = 110

## 3. Gather the data

`load_range` raises on a missing day rather than leaving a silent gap, because a
gap would read downstream as a storm ending and restarting.

In [ ]:
da = load_range(START, END, source=SOURCE, product=PRODUCT)
R  = da.values
print(da.sizes, f"{R.nbytes/1e6:.0f} MB")
print(f"units {da.attrs.get('units', 'mm/hr')}, "
      f"max {np.nanmax(R):.1f}, wet>=1mm/h {100*(R>=1).mean():.1f}%")

### Domain rainfall through the window

`precipitation` is an intensity in mm/hr, so depth per half-hourly step is
`R * 0.5` mm. Omitting the 0.5 is a factor-of-two error.

In [ ]:
from imerg_io import cell_area_m2
from config import DT_HOURS

area = cell_area_m2(da.lat.values)[None, :, None]
vol_per_step = (np.nan_to_num(R) * DT_HOURS / 1000.0 * area).sum(axis=(1, 2)) / 1e9

fig, ax = plt.subplots(figsize=(11, 2.8))
ax.plot(da.time.values, vol_per_step, lw=1)
ax.set_ylabel("km$^3$ per 30 min"); ax.set_xlabel("time (UTC)")
ax.set_title(f"Domain rain volume, total {vol_per_step.sum():.1f} km$^3$")
ax.margins(x=0); fig.tight_layout()

# Timestamps are UTC. Indonesia spans UTC+7 to +9, so the diurnal cycle is offset.
print("peak step (UTC):", str(da.time.values[vol_per_step.argmax()])[:16])

## 4. Why not the published method

ST-CORA thresholds and then labels connected components. Over this domain that
percolates: a single object swallows most of the rain field. The cell below shows
the trade the baseline cannot escape.

In [ ]:
rows = []
for name, lab in [
    ("CCL thr=1.0", label_ccl(R, threshold=1.0, min_voxels=MIN_VOXELS)),
    ("CCL thr=5.0", label_ccl(R, threshold=5.0, min_voxels=MIN_VOXELS)),
    (f"watershed h={PROMINENCE}",
     label_watershed(R, WET_THRESHOLD, PROMINENCE, MIN_VOXELS)),
]:
    c = np.bincount(lab.ravel()); c[0] = 0
    rows.append({"method": name, "storms": int((c > 0).sum()),
                 "largest_object_share": percolation_share(lab),
                 "wet_captured": c.sum() / max((R >= 1.0).sum(), 1)})
pd.DataFrame(rows).round(4)

Read the two columns together. The baseline either keeps the rain and percolates,
or de-percolates by discarding most of it. The watershed partitions the field, so
no object can swallow the domain however connected the rain is.

## 5. Build the catalogue

In [ ]:
df = catalogue.build(START, END, source=SOURCE, product=PRODUCT,
                     wet_threshold=WET_THRESHOLD, prominence=PROMINENCE,
                     min_voxels=MIN_VOXELS)
print(f"{len(df)} storms, {df.total_volume_km3.sum():.1f} km3")
df.sort_values("total_volume_km3", ascending=False).head(10)[[
    "start_time", "end_time", "duration_h", "total_volume_km3",
    "max_intensity_mm_hr", "max_area_km2",
    "start_lat", "start_lon", "end_lat", "end_lon"]]

### Truncated storms

A storm clipped by the edge of the AOI or by the start or end of the window has a
censored duration and volume. They are flagged, never dropped. Any duration or
volume distribution that ignores these flags is biased low.

In [ ]:
print(f"truncated in time : {df.truncated_time.sum():5d} "
      f"({100*df.truncated_time.mean():.1f}%)")
print(f"truncated in space: {df.truncated_space.sum():5d} "
      f"({100*df.truncated_space.mean():.1f}%)")
clean = df[~(df.truncated_time | df.truncated_space)]
print(f"\nduration median, all storms {df.duration_h.median():.1f} h, "
      f"untruncated only {clean.duration_h.median():.1f} h")

## 6. Storm population

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 3.2))
axes[0].hist(clean.duration_h, bins=40)
axes[0].set_xlabel("duration (h)"); axes[0].set_ylabel("storms")
axes[1].hist(np.log10(df.total_volume_km3.clip(lower=1e-6)), bins=40)
axes[1].set_xlabel("log$_{10}$ volume (km$^3$)")
axes[2].hist(df.max_intensity_mm_hr, bins=40)
axes[2].set_xlabel("max intensity (mm/hr), peak voxel")
for a in axes: a.set_yscale("log")
fig.suptitle(f"{len(df)} storms, h={PROMINENCE} mm/hr  "
             f"(duration panel excludes truncated storms)", y=1.04)
fig.tight_layout()

## 7. Tracks

The trackline is the volume-weighted centroid at each timestep. Start and end
location are its first and last vertex.

The AOI includes ocean deliberately: most storms here form over water and move
onto land, so the sea carries the early life of the track.

In [ ]:
top = df.sort_values("total_volume_km3", ascending=False).head(25)

fig, ax = plt.subplots(figsize=(13, 5.2))
ax.imshow(np.nanmean(R, axis=0), origin="lower", cmap="Blues",
          extent=[da.lon.values[0], da.lon.values[-1],
                  da.lat.values[0], da.lat.values[-1]],
          vmax=np.nanpercentile(np.nanmean(R, axis=0), 99))
for _, s in top.iterrows():
    lats = [p[1] for p in s.track]; lons = [p[2] for p in s.track]
    ax.plot(lons, lats, lw=1.4, alpha=.9)
    ax.plot(lons[0], lats[0], "o", ms=4, mfc="none", mec="k", mew=.8)
    ax.plot(lons[-1], lats[-1], "x", ms=5, mec="k", mew=.9)
ax.set_xlabel("lon"); ax.set_ylabel("lat")
ax.set_title(f"25 largest storms by volume. Background: mean rain rate. "
             f"o start, x end")
fig.tight_layout()

## 8. Save

Parquet, with the trackline as WKT so it opens in any GIS, plus a sidecar JSON of
the parameters and the git revision that produced it.

In [ ]:
path = catalogue.save(df, START, END, PRODUCT)
print(path)
df[[c for c in df.columns if c.startswith("param_")]].head(1).T

## Caveats to carry with any result from this notebook

1. **No validation.** No gauge or radar comparison. This is a catalogue, not a
   measurement of skill.
2. **The storm count depends on `PROMINENCE`.** It roughly halves per doubling of
   h. There is no canonical storm count; h selects the scale of system counted.
   Always quote h. What is robust across h is the percolation share and the
   captured fraction.
3. **Times are UTC.** Indonesia spans UTC+7 to +9, and the land-sea diurnal cycle
   is the dominant signal here, so any timing statistic needs an explicit
   convention.
4. **Truncated storms are censored**, not wrong. Filter on the flags before
   quoting any duration or volume distribution.
5. **Runs differ.** Late reads about 9% higher in total volume than Final. Quote
   the run.
6. **A domain aggregate blurs regimes.** Indonesia has several rainfall regimes
   with opposing seasonality; domain-wide numbers hide them. December is the
   wettest month of 2020 domain-wide and January ranks 9th of 12.
